# Validación real de full — FINTERM 0.1
Notebook sin ejecutar. Elegir el kernel H100 preparado según INSTALL.md. Las rutas se toman de variables de entorno, no de un ordenador particular. Este protocolo prueba ejecución, no precisión semántica ni enlaces definitivos.


In [ ]:
import os, sys, json, subprocess
from pathlib import Path
from datetime import datetime, timezone
assert sys.platform != 'win32', 'Usar el servidor H100'
assert sys.version_info[:2] == (3, 11), sys.version
from finterm_linking import Linker, LinkerConfig
from finterm_linking.cli import doctor
from finterm_linking.acceptance import validate_h100_acceptance
PROJECT = Path(os.environ['FINTERM_PROJECT']).expanduser().resolve()
BASE_WORK = Path(os.environ['FINTERM_ACCEPTANCE_WORK']).expanduser().resolve()
QWEN14 = Path(os.environ['FINTERM_QWEN14_CACHE']).expanduser().resolve()
EMBEDDING = Path(os.environ['FINTERM_EMBEDDING_RESOURCES']).expanduser().resolve()
CONTACT = os.environ['FINTERM_WIKIMEDIA_CONTACT']
report = doctor(embedding_resources=EMBEDDING, hub_cache=QWEN14)
assert report['h100_environment']['compatible'], report['h100_environment']
print(sys.executable, report['version'])


In [ ]:
WORK = BASE_WORK / ('full_acceptance_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f'))
WORK.mkdir(parents=True, exist_ok=False)
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True, check=True).stdout
(WORK / 'environment.freeze.txt').write_text(freeze, encoding='utf-8')
config = LinkerConfig(work_dir=WORK, mode='full', similarity_mode='h100', offline=False,
    qwen14_hub_cache=QWEN14, embedding_resource_dir=EMBEDDING, http_contact=CONTACT,
    search_policy='historical', translation_policy='historical', preselection_policy='conservative',
    max_translations=256, max_gate_calls=32, max_definition_calls=16, max_new_embeddings=1024)
linker = Linker(config)
cases = json.loads((PROJECT / 'examples/financial_cases.json').read_text(encoding='utf-8'))
run_dirs = []


## Casos reales y condicionales
Primero se prueba finanzas. Los casos adicionales son consultas naturales sugeridas por evidencia ya presente en los few-shot; sirven para verificar los mecanismos y pueden solaparse con demostraciones, por lo que no son evaluación independiente de calidad. No se inyecta ningún QID, se elimina ninguna descripción ni se cambia la política para forzar DefTerm. Si Wikidata ya aporta texto utilizable, el módulo debe abstenerse de generar.


In [ ]:
for case in cases:
    result = linker.link(case['text'], case['terms'], document_id=case['document_id'], top_k=5)
    run_dirs.append(result.run_dir)
    print(case['document_id'], result.status, result.execution_kind)
    print(json.dumps(result.trace['inferences'], ensure_ascii=False, indent=2))
    print('Top 3:', json.dumps(result.top(3), ensure_ascii=False))
    print('Top 5:', json.dumps(result.top(5), ensure_ascii=False))
    acceptance = validate_h100_acceptance(run_dirs)
    if acceptance['full_runtime_acceptance_passed']:
        break


In [ ]:
acceptance = validate_h100_acceptance(run_dirs)
(WORK / 'h100_full_acceptance.json').write_text(json.dumps(acceptance, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(acceptance['checks'], indent=2))
print('Todas las tareas terminadas:', acceptance['all_requested_tasks_completed'])
assert acceptance['full_runtime_acceptance_passed'], ('NO VALIDADO: faltan capacidades reales. Revisar elegibilidad, pendientes, recursos y presupuestos; añadir casos reales sin forzar decisiones.', WORK)


Si no hubo una traducción válida, un gate positivo y una definición válida de producción con intentos nuevos, el protocolo falla expresamente. Añadir casos genuinos o completar presupuestos pendientes usando las cachés, sin borrar errores ni claims ambiguos. No sustituir esos requisitos por mocks o respuestas manuales.
La aceptación de capacidades no elimina estados PARTIAL ni valida fidelidad, precisión, Top 1 o enlaces definitivos. Los informes muestran ambas cosas por separado.
